# Notebook 02: Signal Processing Tutorial

**AI Power Electronics Diagnostics — IEEE IES Industrial AI Lab**

This notebook is a hands-on tutorial covering the signal processing pipeline:

1. **FFT Analysis** — frequency spectrum, THD computation
2. **STFT Spectrogram** — time-frequency representation for CNN input
3. **Wavelet Analysis** — DWT multi-resolution, CWT scalogram
4. **Harmonic Analysis** — IEEE 519 THD, sideband detection
5. **Feature Extraction** — unified pipeline for ML training

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from datasets.synthetic import InverterFaultSimulator, MotorDriveSimulator
from signal_processing import FFTAnalyzer, STFTSpectrogram, WaveletFeatureExtractor, HarmonicAnalyzer, SignalFeatureExtractor
from visualization import SpectrogramPlotter

plt.rcParams['figure.dpi'] = 100

# Generate test signals
inv_sim = InverterFaultSimulator()
s_healthy, _ = inv_sim.generate('healthy')
s_oc_t1, _   = inv_sim.generate('open_circuit_T1')
s_sc, _       = inv_sim.generate('short_circuit')

F_SAMPLE = 100_000.0
print('Test signals generated.')

## 1. FFT Analysis

In [ ]:
analyzer = FFTAnalyzer(f_sample=F_SAMPLE)

result_h = analyzer.compute(s_healthy[3])  # Channel Ia
result_f = analyzer.compute(s_oc_t1[3])

print(f'Healthy  | Dominant freq: {result_h.dominant_freq:.1f} Hz | THD: {result_h.thd:.2f}%')
print(f'OC Fault | Dominant freq: {result_f.dominant_freq:.1f} Hz | THD: {result_f.thd:.2f}%')

In [ ]:
plotter = SpectrogramPlotter(f_sample=F_SAMPLE)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

mask = result_h.frequencies <= 5000
axes[0].semilogy(result_h.frequencies[mask], result_h.amplitude_spectrum[mask] + 1e-9)
axes[0].set_title('FFT — Healthy Signal (Ia)')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Amplitude (log)')
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(result_f.frequencies[mask], result_f.amplitude_spectrum[mask] + 1e-9, color='#d62728')
axes[1].set_title('FFT — Open-Circuit T1 Fault (Ia)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. STFT Spectrogram

The STFT shows how frequency content evolves over time — critical for detecting transient faults.

In [ ]:
stft = STFTSpectrogram(f_sample=F_SAMPLE, n_fft=512, output_size=(128, 128))

result_spec = stft.compute(s_healthy[3])
print(f'Spectrogram shape: {result_spec.spectrogram.shape}')
print(f'Time resolution: {stft.time_resolution*1000:.2f} ms | Freq resolution: {stft.freq_resolution:.1f} Hz')

# Compare healthy vs short-circuit spectrograms
signals_cmp = {
    'healthy': s_healthy[3],
    'open_circuit_T1': s_oc_t1[3],
    'short_circuit': s_sc[3]
}
fig = plotter.plot_spectrogram_comparison(signals_cmp, n_fft=512, f_max=5000)
plt.show()

## 3. Wavelet Analysis

DWT provides multi-resolution analysis — high-frequency detail at short time scales, low-frequency approximation at long scales.

In [ ]:
wfe = WaveletFeatureExtractor(wavelet='db4', level=5)

dwt_healthy = wfe.dwt_decompose(s_healthy[3])
dwt_fault   = wfe.dwt_decompose(s_oc_t1[3])

print('DWT Sub-band Relative Energies:')
print(f'  Healthy  : {dwt_healthy.relative_energies.round(4)}')
print(f'  OC Fault : {dwt_fault.relative_energies.round(4)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
labels = ['cA5', 'cD5', 'cD4', 'cD3', 'cD2', 'cD1']

axes[0].bar(labels, dwt_healthy.relative_energies * 100, color='#2ca02c')
axes[0].set_title('DWT Sub-band Energy — Healthy')
axes[0].set_ylabel('Relative Energy (%)')
axes[0].grid(True, axis='y', alpha=0.3)

axes[1].bar(labels, dwt_fault.relative_energies * 100, color='#d62728')
axes[1].set_title('DWT Sub-band Energy — Open-Circuit T1')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Harmonic Analysis

In [ ]:
harm_analyzer = HarmonicAnalyzer(f_sample=F_SAMPLE, f_fund_nominal=50.0)

# Motor drive signals have more interesting harmonic content
motor_sim = MotorDriveSimulator()
s_itsc, _ = motor_sim.generate('itsc')
s_motor_h, _ = motor_sim.generate('healthy')

harm_h = harm_analyzer.analyze(s_motor_h[0])
harm_f = harm_analyzer.analyze(s_itsc[0])

print('Harmonic Analysis Results:')
print(f'  Healthy  | THD-F: {harm_h.thd_f:.2f}% | inter-harmonics: {len(harm_h.inter_harmonics)}')
print(f'  ITSC     | THD-F: {harm_f.thd_f:.2f}% | inter-harmonics: {len(harm_f.inter_harmonics)}')

fig = plotter.plot_harmonics(harm_f.harmonics, f_fund=50.0, thd_f=harm_f.thd_f,
                              title='ITSC Fault Harmonic Spectrum')
plt.show()

## 5. Unified Feature Extraction Pipeline

In [ ]:
# Generate a small dataset
X_raw, y = inv_sim.generate_dataset(n_per_class=20, window_size=1024)
print(f'Raw dataset: X={X_raw.shape}, y={y.shape}')

# Extract hand-crafted features
extractor_feat = SignalFeatureExtractor(f_sample=F_SAMPLE, output_mode='features')
X_feat = extractor_feat.transform_batch(X_raw)
print(f'Feature matrix: X={X_feat.shape}  ({X_feat.shape[1]} features per sample)')

# Extract spectrograms for CNN
extractor_spec = SignalFeatureExtractor(f_sample=F_SAMPLE, output_mode='spectrogram',
                                         spectrogram_size=(64, 64))
X_spec = extractor_spec.transform_batch(X_raw)
print(f'Spectrogram tensor: X={X_spec.shape}')

print('\nNotebook 02 complete.')